In [ ]:
!pip install -U datasets tokenizers transformers peft bitsandbytes

#Instruction Tuning

In [ ]:
!pip install -q more-itertools transformers bitsandbytes peft datasets

In [ ]:
!pip install -U datasets

In [ ]:
import json
import re
from pathlib import Path
import xml.etree.ElementTree as ET

def clean_text(raw):
    """
    Cleans the raw text by removing artifacts and extracting a representative excerpt.
    """
    # Remove artifacts and control tokens
    raw = re.sub(r'##LT(Line|Figure)##', '', raw)
    raw = re.sub(r'\(cid:\d+\)', '', raw)
    raw = re.sub(r'[−–—]', '-', raw)
    raw = re.sub(r'\s+', ' ', raw)

    # Remove overlong garbage lines (math dumps, chart labels etc)
    raw = re.sub(r'\b(?:Fig\.?|Table|Frequency|Hz|nm|dB|Time|Error|Estimated)\b.*?(?=\.)', '', raw, flags=re.IGNORECASE)

    # Try to find a representative excerpt between 2 and 4 sentences
    candidates = re.split(r'(?<=[.?!])\s+', raw.strip())
    excerpt = ' '.join(candidates[:3]).strip()

    return excerpt if len(excerpt) > 20 else raw[:300].strip()

def inject_text_tag_and_strip(xml_str: str, excerpt: str) -> str:
    """
    Parses the XML string, removes <filename> elements, injects <text> tags, and returns the modified XML.
    """
    root = ET.fromstring(xml_str)

    # Remove each <filename> element
    for fname in root.findall("filename"):
        root.remove(fname)

    # Inject the <text> tag under each <object> after its <name>
    for obj in root.findall("object"):
        name = obj.find("name")
        if name is not None:
            txt = ET.Element("text")
            txt.text = excerpt
            idx = list(obj).index(name)
            obj.insert(idx + 1, txt)

    # Serialize only the *children* of <annotation>, dropping the wrapper itself
    inner = []
    for child in root:
        inner.append(ET.tostring(child, encoding="unicode"))
    return "".join(inner)

def convert_and_clean(in_path, out_path):
    """
    Processes the input JSONL file, cleans and formats each example, and writes to the output JSONL file.
    """
    print(f"Processing: {in_path}")
    count = 0
    with open(in_path) as f_in, open(out_path, "w") as f_out:
        for line in f_in:
            try:
                obj = json.loads(line)
                excerpt = clean_text(obj["text"])
                new_xml = inject_text_tag_and_strip(obj["xml"], excerpt)
                prompt = (
                    "You are a helpful assistant. Write a response that appropriately completes the request.\n\n"
                    f"### Input:\n{excerpt}\n\n### Response:"
                )
                clean = {
                    "prompt": prompt,
                    "output": new_xml
                }
                f_out.write(json.dumps(clean) + "\n")
                count += 1
            except Exception as e:
                print(f"Skipping line due to error: {e}")
    print(f"Wrote {count} cleaned examples to {out_path}")

def validate_jsonl(file_path, n=3):
    """
    Validates the JSONL file by checking for required keys and printing sample entries.
    """
    print(f"\n Validating: {file_path}")
    samples = []
    with open(file_path) as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
                assert "prompt" in obj and "output" in obj
                samples.append(obj)
            except Exception as e:
                print(f"⚠️ Line {i} is malformed: {e}")
            if len(samples) >= n:
                break
    for s in samples:
        print("\n--- Sample ---")
        print(f" Prompt: {s['prompt'][:150]}...")
        print(f" Output: {s['output'][:150]}...")


# Paths for your actual files (update if needed)
convert_and_clean("/content/500k_train.with_text.jsonl", "/content/train_instruct.jsonl")
convert_and_clean("/content/500k_valid.with_text.jsonl", "/content/valid_instruct.jsonl")
convert_and_clean("/content/500k_test.with_text.jsonl", "/content/test_instruct.jsonl")

# Run it on validation
validate_jsonl("/content/train_instruct.jsonl")
validate_jsonl("/content/valid_instruct.jsonl")
validate_jsonl("/content/test_instruct.jsonl")


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MAX_TOKENIZATION_LENGTH = 512

# Load pretrained tokenizer from base model
tokenizer = AutoTokenizer.from_pretrained("princeton-nlp/Sheared-LLaMA-1.3B-ShareGPT", use_fast=True)
tokenizer.add_special_tokens({
    "pad_token": tokenizer.eos_token  # Add only <pad> if needed
})

# Load raw JSONL files
data_files = {
    "validation": "/content/valid_instruct.jsonl",
    "test":       "/content/test_instruct.jsonl",
    "train":      "/content/train_instruct.jsonl"
}
raw = load_dataset("json", data_files=data_files)

# Tokenization function
BOS_ID = tokenizer.bos_token_id   # 1
EOS_ID = tokenizer.eos_token_id   # 2
PAD_ID = tokenizer.pad_token_id   # 2

def tokenize_fn(examples):
    prompts = examples["prompt"]
    outputs = examples["output"]

    # Tokenize prompts
    model_inputs = tokenizer(
        prompts,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TOKENIZATION_LENGTH - 2,
        padding=False,
    )

    # Tokenize outputs
    labels = tokenizer(
        outputs,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TOKENIZATION_LENGTH - 1,
        padding=False,
    )

    input_ids, attention_mask, label_ids = [], [], []
    for inp_ids, attn, lbl_ids in zip(model_inputs["input_ids"],
                                      model_inputs["attention_mask"],
                                      labels["input_ids"]):
        # Add BOS token
        inp_ids = [BOS_ID] + inp_ids
        attn   = [1]       + attn

        # Add EOS token
        inp_ids = inp_ids + [EOS_ID]
        attn    = attn    + [1]
        lbl_ids = lbl_ids + [EOS_ID]

        # Pad sequences
        pad_len = MAX_TOKENIZATION_LENGTH - len(inp_ids)
        inp_ids += [PAD_ID] * pad_len
        attn    += [0]      * pad_len

        # Create labels with -100 for prompt and padding tokens
        labels_full = [-100] * (len(inp_ids) - len(lbl_ids)) + lbl_ids

        input_ids.append(inp_ids)
        attention_mask.append(attn)
        label_ids.append(labels_full)

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         label_ids,
    }



# Apply tokenizer
tokenized = raw.map(tokenize_fn, batched=True, remove_columns=raw["validation"].column_names)

# ─── Sanity-check BOS/EOS/padding ────────────────────────────────────────
from itertools import islice

# pick first 3 examples from each split
for split in ["train", "validation", "test"]:
    print(f"\n=== {split.upper()} SAMPLE ===")
    ds = tokenized[split]
    for idx in range(3):
        ex = ds[idx]
        ids, mask, labs = ex["input_ids"], ex["attention_mask"], ex["labels"]

        # find where generation actually starts (labels != -100)
        gen_pos = [i for i,v in enumerate(labs) if v != -100]
        start, end = (gen_pos[0], gen_pos[-1]+1) if gen_pos else (None, None)

        print(f"\n- Example {idx}")
        print("Input slice:      ", ids[:10], "...", ids[-10:])
        print("Attention slice:  ", mask[:10], "...", mask[-10:])
        print("Labels slice:     ", labs[start-5:start] if start else None, labs[start:end], labs[end:end+5] if end else None)
        print("Decoded full inp: ", tokenizer.decode(ids, skip_special_tokens=False))
        print("Decoded target:   ", tokenizer.decode([v for v in labs if v not in (-100, PAD_ID)], skip_special_tokens=False))


val_ds  = tokenized["validation"]
test_ds = tokenized["test"]
train_ds = tokenized["train"]


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# Enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# BitsAndBytes config for quantized model
quant_cfg = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    "princeton-nlp/Sheared-LLaMA-1.3B-ShareGPT",
    quantization_config=quant_cfg,
    torch_dtype=torch.float16,
    use_cache=True
)

# Add LoRA layers
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_cfg)

In [ ]:
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import TrainerCallback

class PlateauLRCallback(TrainerCallback):
    """
    Reduce the LR by `factor` if `eval_loss` doesn't improve for `patience` evaluations.
    """
    def __init__(self, factor=0.5, patience=1, mode="min"):
        super().__init__()
        self.factor = factor
        self.patience = patience
        self.mode = mode
        self.scheduler = None

    def on_train_begin(self, args, state, control, **kwargs):
        # grab the HF optimizer from kwargs
        optimizer = kwargs.get("optimizer")
        if optimizer is None:
            raise ValueError("No optimizer found in callback kwargs")
        self.scheduler = ReduceLROnPlateau(
            optimizer,
            mode=self.mode,
            factor=self.factor,
            patience=self.patience,
            verbose=True,
        )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        # step the scheduler on every eval
        if metrics is not None and "eval_loss" in metrics:
            self.scheduler.step(metrics["eval_loss"])


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="instruct_tuned_subset",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    fp16=True,
    optim="adamw_bnb_8bit",
    learning_rate=2e-5,
    num_train_epochs=5,
    eval_strategy="epoch",  # run eval each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    callbacks=[PlateauLRCallback(factor=0.5, patience=1)],
)

trainer.train()
print("Model saved to", training_args.output_dir)


#Model Upload

In [ ]:
!git config --global user.email "BloggSeb@gmail.com"
!git config --global user.name "Seb Blogg"


In [ ]:
# install the HF helper library if you haven’t already
!pip install -q huggingface_hub

from huggingface_hub import login, create_repo, Repository

# 1.1) Log in (set HF_TOKEN in your Colab env)
login()


In [ ]:
from huggingface_hub import create_repo, Repository, create_inference_endpoint
import os
import shutil

# 3️⃣ Set your repo ID
repo_id = "Sebulah/JOLT_Llama"

# 4️⃣ Create a private repo on the Hub (no-op if it already exists)
create_repo(repo_id, private=True)

# 5️⃣ Clone it locally
repo = Repository(local_dir="JOLT_llama_hf", clone_from=repo_id)

# 6️⃣ Copy your checkpoint artifacts into that folder
src = "/content/finetuned_subset/checkpoint-40000"
dst = os.path.join("JOLT_llama_hf", "checkpoint-40000")
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# 7️⃣ Push your model, tokenizer, adapter, etc.
repo.push_to_hub(commit_message="Add fine-tuned Sheared-LLaMA 1.3B + LoRA")

print("Done! Your model is live at", repo_id)


#Grid Search

In [ ]:
import itertools
from difflib import SequenceMatcher
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
from peft import PeftModel
from tqdm.auto import tqdm

# ─── CONFIG ────────────────────────────────────────────────────────────────
BASE_MODEL = "princeton-nlp/Sheared-LLaMA-1.3B"
CHECKPOINTS = [
    "/content/FineTune_50k",
    "/content/Instruct_50k",
    "/content/Instruct_100k",
]

INSTR = ''' Here’s an EXAMPLE of the desired input/output:

"instruction": "Generate XML annotation for the following excerpt:",
"input": "SINGULAR SOLUTIONS TO THE KAPUSTIN-WITTEN EQUATIONS 11 Proof. By a direct computation. In this subsection, we will prove that we can obtain the t\u2019Hooft solution of Yang-Mills equation from the equations (7).",
"output": "<size><width>773</width><height>1000</height><depth>3</depth></size><object><name>paragraph</name><text>SINGULAR SOLUTIONS TO THE KAPUSTIN-WITTEN EQUATIONS 11 Proof. By a direct computation. In this subsection, we will prove that we can obtain the t\u2019Hooft solution of Yang-Mills equation from the equations (7).</text><bndbox><xmin>113</xmin><ymin>251</ymin><xmax>658</xmax><ymax>291</ymax></bndbox></object>"}


Now, generate XML for this new excerpt:
"instruction": "Generate XML annotation for the following excerpt:",
"input": '''

excerpt = "The interferometric baselines and GRAVITY astrometric . Lacoura, F. Eisenhauerb, S."
gold_xml = "<size><width>773</width><height>1000</height><depth>3</depth></size><object><name>paragraph</name><text>The interferometric baselines and GRAVITY astrometric . Lacoura, F. Eisenhauerb, S.</text><bndbox><xmin>78</xmin><ymin>511</ymin><xmax>693</xmax><ymax>696</ymax></bndbox></object>"
prompt = f"{INSTR}\n{excerpt}"

# ─── CONSTRAINT PROCESSOR ──────────────────────────────────────────────────
class RestrictedTokenProcessor(LogitsProcessor):
    def __init__(self, tokenizer, max_digits=3, banned_tokens=None):
        self.tokenizer = tokenizer
        self.max_digits = max_digits
        self.digit_tokens = {tokenizer.convert_tokens_to_ids(str(d)) for d in range(10)}
        self.banned_token_ids = {tokenizer.convert_tokens_to_ids(t) for t in (banned_tokens or [])}

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        # Ban raw characters like "<" and ">"
        for token_id in self.banned_token_ids:
            scores[0, token_id] = -float("inf")

        # Block more than N consecutive digits
        if input_ids.shape[1] >= self.max_digits:
            last_ids = input_ids[0, -self.max_digits:]
            if all(tok.item() in self.digit_tokens for tok in last_ids):
                for d in self.digit_tokens:
                    scores[0, d] = -float("inf")

        return scores

# ─── UTILS ─────────────────────────────────────────────────────────────────
def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

def build_prefix_filter(tokenizer):
    TAG_TOKENS = {
        t: tokenizer.convert_tokens_to_ids(t)
        for t in ["<size>","</size>","<width>","</width>","<height>","</height>",
                  "<depth>","</depth>","<object>","</object>","<name>","</name>",
                  "<text>","</text>","<bndbox>","</bndbox>"]
    }
    DIGITS = {str(d): tokenizer.convert_tokens_to_ids(str(d)) for d in range(10)}

    def prefix_allowed(batch_id, input_ids):
        text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        if text.endswith("</name>"):
            return [TAG_TOKENS["<text>"]]
        if text.endswith("</text>"):
            return [TAG_TOKENS["<bndbox>"]]
        return list(TAG_TOKENS.values()) + list(DIGITS.values())

    return prefix_allowed

# ─── MAIN EXECUTION ────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
prompt_len = inputs["input_ids"].shape[-1]

for ckpt in CHECKPOINTS:
    print(f"\n Searching LoRA checkpoint: {Path(ckpt).name}")
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device).eval()
    model = PeftModel.from_pretrained(base_model, ckpt).to(device).eval()

    best_score = -1.0
    best_params = None
    best_out = None

    param_grid = {
        "num_beams":            [1, 4,8],
        "no_repeat_ngram_size": [2, 4, 6],
        "repetition_penalty":   [0.5, 1.1, 1.5],
        "max_new_tokens":       [50, 80, 100],
        "prefix_allowed_tokens_fn": [build_prefix_filter(tokenizer)],
    }

    for combo in tqdm(list(itertools.product(*param_grid.values())), desc=f"Grid for {Path(ckpt).name}"):
        params = dict(zip(param_grid.keys(), combo))
        with torch.no_grad():
            output = model.generate(
                **inputs,
                pad_token_id=tokenizer.eos_token_id,
                logits_processor=LogitsProcessorList([
                    RestrictedTokenProcessor(tokenizer, max_digits=3, banned_tokens=["<", ">"])
                ]),
                **params
            )

        gen_ids = output[0][prompt_len:]
        cand = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        score = similarity(cand, gold_xml)

        if score > best_score:
            best_score, best_params, best_out = score, params, cand

    print(f"\n Best score for {Path(ckpt).name}: {best_score:.4f}")
    print(f" Params: {best_params}")
    print(f" Output:\n{best_out}\n")

    del model, base_model
    torch.cuda.empty_cache()


In [ ]:
!pip install evaluate rich rouge_score zss

In [ ]:
import xml.etree.ElementTree as ET
from datasets import load_dataset
from evaluate import load as load_metric
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from peft import PeftModel

# ─── CONFIG ────────────────────────────────────────────────────────────
BASE_MODEL  = "princeton-nlp/Sheared-LLaMA-1.3B"
CHECKPOINTS = [
    "/content/FineTune_50k",
    "/content/Instruct_50k",
    "/content/Instruct_100k",
]
TEST_JSONL  = "test_instruct.jsonl"
MAX_NEW     = 200
BATCH_SIZE  = 32

# ─── CUSTOM DIGIT CONSTRAINT ───────────────────────────────────────────
class MaxConsecutiveDigitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, max_digits=3):
        self.tokenizer = tokenizer
        self.max_digits = max_digits
        self.digit_tokens = {tokenizer.convert_tokens_to_ids(str(d)) for d in range(10)}

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if input_ids.shape[1] < self.max_digits:
            return scores

        last_ids = input_ids[0, -self.max_digits:]
        count = 0
        for token_id in reversed(last_ids.tolist()):
            if token_id in self.digit_tokens:
                count += 1
            else:
                break

        if count >= self.max_digits:
            for d in self.digit_tokens:
                scores[0, d] = -float("inf")

        return scores

# ─── SHARED SETUP ───────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tok.pad_token = tok.eos_token
bnb_cfg = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=False
)

# load & subsample test set once
ds = load_dataset("json", data_files={"test": TEST_JSONL})["test"]
ds = ds.shuffle(seed=42).select(range(125))

# metrics
bleu  = load_metric("bleu")
rouge = load_metric("rouge")
def well_formed(x):
    try: ET.fromstring(x); return True
    except: return False

def generate_batchwise(model, batch, use_digit_constraint=False):
    prompts = [f"{i}\n{inp}" for i, inp in zip(batch["instruction"], batch["input"])]
    inputs = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
    inputs.pop("token_type_ids", None)

    logits_processor = None
    if use_digit_constraint:
        logits_processor = LogitsProcessorList([MaxConsecutiveDigitsProcessor(tok, max_digits=3)])

    outs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW,
        do_sample=False,
        num_beams=1,
        pad_token_id=tok.eos_token_id,
        logits_processor=logits_processor
    )
    return tok.batch_decode(outs, skip_special_tokens=True)

# ─── EVALUATE BASE MODEL ─────────────────────────────────────────────────
print(" Evaluating base model…")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    torch_dtype=torch.float16,
    device_map="auto",).eval()

wf_b, preds_b, refs_b = 0, [], []
for i in tqdm(range(0, len(ds), BATCH_SIZE), desc="base batches"):
    batch = ds[i : i + BATCH_SIZE]
    batch_preds = generate_batchwise(base, batch)
    for p, gold in zip(batch_preds, batch["output"]):
        preds_b.append(p.strip())
        refs_b.append(gold)
        if well_formed(p): wf_b += 1

bleu_b  = bleu.compute(predictions=preds_b, references=[[r] for r in refs_b])
rouge_b = rouge.compute(predictions=preds_b, references=refs_b)
print(f"\n Base well-formed: {wf_b}/{len(ds)}")
print(f" BLEU {bleu_b['bleu']:.4f}    ROUGE1 {rouge_b['rouge1']:.4f}")

del base
torch.cuda.empty_cache()

# ─── EVALUATE EACH LoRA CHECKPOINT ────────────────────────────────────────
for ckpt in CHECKPOINTS:
    print(f"\n Loading fresh base + adapter {ckpt.split('/')[-1]}")
    fresh_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_cfg,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(fresh_base, ckpt, device_map="auto").eval().to(device)

    wf, preds, refs = 0, [], []
    for i in tqdm(range(0, len(ds), BATCH_SIZE), desc=f"LoRA {ckpt.split('/')[-1]}"):
        batch = ds[i : i + BATCH_SIZE]
        batch_preds = generate_batchwise(model, batch, use_digit_constraint=True)
        for p, gold in zip(batch_preds, batch["output"]):
            preds.append(p.strip())
            refs.append(gold)
            if well_formed(p):
                wf += 1

    bleu_r  = bleu.compute(predictions=preds, references=[[r] for r in refs])
    rouge_r = rouge.compute(predictions=preds, references=refs)

    pct = wf / len(ds)
    print(f"\n Well‐formed: {wf}/{len(ds)} = {pct:.1%}")
    print(f" BLEU:  {bleu_r['bleu']:.4f}")
    formatted_rouge = {
        'rouge1':    f"{rouge_r['rouge1']:.4f}",
        'rouge2':    f"{rouge_r['rouge2']:.4f}",
        'rougeL':    f"{rouge_r['rougeL']:.4f}",
        'rougeLsum': f"{rouge_r['rougeLsum']:.4f}",
    }
    print(f" ROUGE: {formatted_rouge}")

    del model, fresh_base
    torch.cuda.empty_cache()


In [ ]:
import random
from rich import print as rprint  

# How many examples to show
NUM_EXAMPLES = 5
EXAMPLE_INDICES = random.sample(range(len(ds)), NUM_EXAMPLES)

# Gather all predictions from all evaluated models
all_preds = {
    "Base": preds_b,
}

# Add LoRA predictions with model names
for ckpt, preds_lora in zip(CHECKPOINTS, [preds]):
    model_name = ckpt.split("/")[-1]  # e.g., "Finetune_50k"
    all_preds[model_name] = preds_lora

# Print predictions side by side
print("\n\nModel Predictions Comparison\n")
for idx in EXAMPLE_INDICES:
    instruction = ds[idx]["instruction"]
    input_text = ds[idx]["input"]
    gold = ds[idx]["output"]

    rprint(f"[bold cyan] Example Index: {idx}[/bold cyan]")
    rprint(f"[bold yellow]Instruction:[/bold yellow] {instruction}")
    rprint(f"[bold yellow]Input:[/bold yellow] {input_text}\n")
    rprint(f"[bold green] Reference Output:[/bold green] {gold}\n")

    for model_name, predictions in all_preds.items():
        rprint(f"[bold magenta]{model_name} Prediction:[/bold magenta] {predictions[idx]}\n")

    print("=" * 100)
